## Comparison of Machine Learning Models for Time Series Analysis

### Overview

In this analysis, we provide a comparison of prediction accuracy of the forecasting systems introduced Module 5, specifically, Prophet, Orbit, SilverKite and LSTM. The comparison will be provided over multiple testing periods using two approaches.

### Setup

We set up first the analysis to be run under the main ISYE6402 repository Module 5. Then we import the libraries requried for this analysis.

In [ ]:
! pip install neuralforecast

In [ ]:
!pip install prophet==1.1.5 "numpy<2.0" cmdstanpy==1.2.0

In [ ]:
! pip install greykite

In [ ]:
! pip install orbit-ml

In [ ]:
! pip install "holidays==0.24.0"

In [ ]:
import warnings
warnings.filterwarnings("ignore")

# Set the environment folder to be within the ISYE6402Main
#import sys
#import os
#from pathlib import Path
#cwd = Path.cwd()
#initCwd = Path.cwd()
#while cwd != Path('/'):
#    if cwd.name == 'ISyE6402Main':
#        os.chdir(cwd)
#        os.chdir('Module5')
#        sys.path.insert(0, str(cwd))
#        break
#    cwd = cwd.parent
#if cwd.name != 'ISyE6402Main':
#    raise Exception(f"Unexpected working directory {initCwd}. Please check the correct directory.")

# Load Python Packages
#import time
#import itertools
#import random
#from collections import defaultdict
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# === Plotting ===
#from matplotlib.ticker import FuncFormatter
#from prophet.plot import add_changepoints_to_plot
#from orbit.diagnostics.plot import plot_predicted_data

# === Forecast Models ===
from prophet import Prophet
from statsmodels.tsa.statespace.sarimax import SARIMAX
from orbit.models import DLT
#from orbit.utils.dataset import load_iclaims
from prophet.diagnostics import cross_validation

#import holidays
#import holidays_ext

#from greykite.common.data_loader import DataLoader
from greykite.framework.templates.forecaster import Forecaster
from greykite.framework.templates.model_templates import ModelTemplateEnum
from greykite.framework.templates.autogen.forecast_config import (
    ForecastConfig,
    MetadataParam,
    ModelComponentsParam,
    EvaluationPeriodParam
)
from greykite.framework.utils.result_summary import summarize_grid_search_results
from greykite.sklearn.cross_validation import RollingTimeSeriesSplit

# === LSTM ===
from neuralforecast.models import LSTM
from neuralforecast.core import NeuralForecast
from neuralforecast.losses.pytorch import DistributionLoss

# === Forecast Evaluation ===
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    #mean_absolute_percentage_error
)


#### Prediction Accuracy Measures

The function below is used to evaluate the prediction accuracy across multiple prediction error measures.

In [ ]:
# ========== evaluation methods ==========
def evaluate_performance(true, pred, model_name="Model"):
    mspe = mean_squared_error(true, pred)
    mae = mean_absolute_error(true, pred)
    mape = np.mean(np.abs((true - pred) / (true + 1e-6)))
    pm = (np.sum((true - pred) ** 2) / np.sum((true - np.mean(true)) ** 2))

    print(f"=== {model_name} Performance ===")
    print(f"MSPE: {mspe:.4f}")
    print(f"MAE:  {mae:.4f}")
    print(f"MAPE: {mape:.4f}")
    print(f"PM:   {pm:.4f}")
    print("-" * 40)

#### Importing data

We import data from the 'Data' folder available with the Module 5 Jupyter notebooks.

In [ ]:
# Load CSV data
df = pd.read_csv('BTCUSD.csv')
#df = pd.read_csv('BTCUSD_historical.csv')

# Convert the 'Date' column to datetime format
df['Date'] = pd.to_datetime(df['Date'])

# Define the start and end dates for filtering
start_date = pd.to_datetime('2019-04-01')
end_date = pd.to_datetime('2025-04-01')

# Keep only rows where Date is between 2019-04-01 and 2025-04-01 (inclusive)
df = df[(df['Date'] >= start_date) & (df['Date'] <= end_date)]

# Keep only the Date and Close columns, rename 'Close' to 'y' for modeling
df = df[['Date', 'Close']].rename(columns={'Close': 'y'}).sort_values('Date').reset_index(drop=True)


In [ ]:
# ========== Check head and tail ==========
print("First 5 rows:")
print(df.head())
print("\nLast 5 rows:")
print(df.tail())

# ========== Split data into training and testing sets ==========
# Use all except the last 7 rows as training data
train_data = df.iloc[:-7].reset_index(drop=True)

# Use the last 7 rows as test data
test_data = df.iloc[-7:].reset_index(drop=True)


### Method 1: Expanding Cross Validation

The method below applies the idea of cross validation to time series. Specifically, we consider the test period to be the last months of the time series to be predicted and predict one prediction period (e.g. one day, one week) at a time on a rolling manner. The first prediction is for the first prediction period of the test period (e.g. most recent three months); that means, we fit the model with the train data including the time series data without the months in the test period and predict the first prediction period of the test period. Once the first prediction period is predicted we append the observed data (not the predicted data!) for that prediction period to the train data and predict the second prediction period in the test period. That is again fit the model with the updated train data and apply it to predict the second prediction period. We continue with predicting the third prediction period by updating again the train data (adding the observed data from the second prediction period in the  test period) and fit the model on the updated train data then apply the fitted model to oobtain the predictions of the third prediction. We continue to do so until we predict all prediction periods (e.g. all weeks) in the test period.

We apply this approach for all models considered in this module as well as the SARIMA model introduced in Module 2 for a more comprehensive comparison.

The function below applies this concept more generally. We can use it for any number of months as the test period and for any time horizon for the prediction period.

In [ ]:
def run_expanding_cv_forecast_mult(df, forecast_horizon, test_period_months, label):
    df = df.sort_values('Date').reset_index(drop=True)
    cutoff_date = df['Date'].max() - pd.DateOffset(months=test_period_months)
    initial_train_end_idx = df[df['Date'] < cutoff_date].index[-1]
    iteration_log = []
    dlt_preds = []
    prophet_preds = []
    silverkite_preds = []
    sarima_preds = []
    lstm_preds = []
    train_end_idx = initial_train_end_idx
    iteration = 1

    while (train_end_idx + forecast_horizon) < len(df):
        train_df = df.iloc[:train_end_idx + 1].copy()
        predict_window = df.iloc[train_end_idx + 1: train_end_idx + 1 + forecast_horizon].copy()
        print(f"Iteration {iteration}: Train = {train_df['Date'].min().date()} ~ {train_df['Date'].max().date()}")

        # ----- Orbit DLT Model implementation-----
        dlt = DLT(
            seasonality=None,
            response_col='y',
            date_col='Date',
            estimator='stan-map',
            seed=8888,
            global_trend_option='loglinear',
            n_bootstrap_draws=100,
        )
        dlt.fit(train_df)
        dlt_forecast = dlt.predict(predict_window)
        dlt_preds.append(dlt_forecast[['Date', 'prediction']].rename(columns={'prediction': 'dlt_pred'}))

        # ----- Prophet Model Implementation -----
        changepoints = ['2020-03-12', '2020-12-16', '2021-05-19', '2021-11-10',
                    '2022-05-09', '2022-11-11', '2023-03-13', '2024-12-05']
        train_df_prophet = train_df.rename(columns={'Date': 'ds', 'y': 'y'})
        predict_window_prophet = predict_window.rename(columns={'Date': 'ds'})
        prophet = Prophet(
            yearly_seasonality=False,
            weekly_seasonality=False,
            changepoints=changepoints
        )
        prophet.fit(train_df_prophet)
        forecast = prophet.predict(predict_window_prophet)
        prophet_forecast = pd.DataFrame({
            'Date': forecast['ds'],
            'prophet_pred': forecast['yhat']
        })
        prophet_preds.append(prophet_forecast)

        # ----- SARIMA Model Implementation-----
        sarima_train = train_df.set_index("Date")["y"]
        try:
            sarima_model = SARIMAX(
                sarima_train,
                order=(1, 1, 0),
                seasonal_order=(1, 0, 1, 12),
                enforce_stationarity=False,
                enforce_invertibility=False,
                validate_specification=False
            ).fit(disp=False)

            sarima_forecast = sarima_model.forecast(steps=forecast_horizon)
            sarima_pred_df = predict_window[['Date']].copy()
            sarima_pred_df['sarima_pred'] = sarima_forecast.values
            sarima_preds.append(sarima_pred_df)
        except Exception as e:
            print(f"Failed SARIMA on iteration {iteration}: {e}")


        # ----- NeuralForecast LSTM Implementation -----
        try:
            lstm_train = train_df.rename(columns={'Date': 'ds'})
            lstm_train['unique_id'] = 'ts1'
            lstm_train = lstm_train[['unique_id', 'ds', 'y']]

            test_df = predict_window.rename(columns={'Date': 'ds'})
            test_df['unique_id'] = 'ts1'
            test_df = test_df[['unique_id', 'ds']]

            lstm_model = LSTM(
                h=forecast_horizon,
                input_size=-1,
                loss=DistributionLoss(distribution='Normal', level=[90, 95]),
                scaler_type='robust',
                encoder_n_layers=3,
                encoder_hidden_size=512,
                decoder_hidden_size=512,
                decoder_layers=2,
                max_steps=200
            )

            nf = NeuralForecast(models=[lstm_model],freq='D')
            nf.fit(df=lstm_train)
            lstm_forecast = nf.predict(futr_df=test_df)
            lstm_pred_df = lstm_forecast.reset_index()
            lstm_pred_df = lstm_pred_df.rename(columns={'ds': 'Date', 'LSTM': 'lstm_pred'})
            lstm_preds.append(lstm_pred_df[['Date', 'lstm_pred']])
        except Exception as e:
            print(f"Failed LSTM on iteration {iteration}: {e}")


        # --- Silverkite Model Implementation ---
        df_combined_cp = train_df.rename(columns={"Date": "ts"})
        silverkite_config = ForecastConfig(
            model_template="SILVERKITE",
            forecast_horizon=forecast_horizon,
            coverage=0.95,
            metadata_param=MetadataParam(time_col="ts", value_col="y", freq="D"),
            model_components_param=ModelComponentsParam(
                growth={"growth_term": "quadratic"},
                changepoints={"changepoints_dict": {
                    "method": "auto",
                    "regularization_strength": 0.5,
                    "no_changepoint_proportion_from_end": 0.0,
                    "potential_changepoint_n": 20,
                    "resample_freq": "7D"
                },
                "seasonality_changepoints_dict": dict(
                    potential_changepoint_distance="30D",
                    seasonality_components_df=pd.DataFrame({
                        "name": ["tow", "conti_year"],
                        "period": [7.0, 1.0],
                        "order": [4, 6],
                        "seas_names": ["weekly", "yearly"]})
                )},
                seasonality={
                    "yearly_seasonality": True,
                    "quarterly_seasonality": False,
                    "monthly_seasonality": False,
                    "weekly_seasonality": False,
                    "daily_seasonality": False,
                },
                autoregression={"autoreg_dict": {"lag_dict": {"orders": [1, 2, 3, 7]}}},
                uncertainty={"uncertainty_dict": {"uncertainty_method": "simple_conditional_residuals"}},
                custom={
                    "feature_sets_enabled": True,
                    "extra_pred_cols": [],
                    "fit_algorithm_dict": {"fit_algorithm": "ridge"}
                }
            )
        )
        forecaster = Forecaster()
        silverkite_result = forecaster.run_forecast_config(df=df_combined_cp, config=silverkite_config)
        sk_df = silverkite_result.forecast.df.rename(columns={"ts": "Date", "forecast": "silverkite_pred"})
        sk_df = sk_df[sk_df['Date'].isin(predict_window['Date'])]
        silverkite_preds.append(sk_df[['Date', 'silverkite_pred']])

        iteration_log.append({
            "Iteration": iteration,
            "Train Start": train_df['Date'].min().date(),
            "Train End": train_df['Date'].max().date(),
            "Predict Start": predict_window['Date'].min().date(),
            "Predict End": predict_window['Date'].max().date()
        })

        train_end_idx += forecast_horizon
        iteration += 1

   # ===== Combine results =====
    dlt_result = pd.concat(dlt_preds, ignore_index=True)
    prophet_result = pd.concat(prophet_preds, ignore_index=True)
    silverkite_result = pd.concat(silverkite_preds, ignore_index=True)
    sarima_result = pd.concat(sarima_preds, ignore_index=True)
    lstm_result = pd.concat(lstm_preds, ignore_index=True)
    merged = df[['Date', 'y']] \
        .merge(dlt_result, on='Date', how='right') \
        .merge(prophet_result, on='Date', how='right') \
        .merge(silverkite_result, on='Date', how='right') \
        .merge(sarima_result, on='Date', how='right') \
        .merge(lstm_result, on='Date', how='right')

    # ===== Filter for dates after 2025 =====
    forecast_start_date = merged['Date'][merged['dlt_pred'].notna()].min()
    plot_df = merged[merged['Date'] >= forecast_start_date]

    # ===== Plot comparison =====
    plt.figure(figsize=(14, 6))
    plt.plot(plot_df['Date'], plot_df['y'], label='Actual', color='black')
    plt.plot(plot_df['Date'], plot_df['dlt_pred'], label='DLT Forecast', linestyle='--')
    plt.plot(plot_df['Date'], plot_df['prophet_pred'], label='Prophet Forecast', linestyle=':')
    plt.plot(plot_df['Date'], plot_df['silverkite_pred'], label='Silverkite Forecast', linestyle='-.')
    plt.plot(plot_df['Date'], plot_df['sarima_pred'], label='SARIMA Forecast', linestyle='--', color='purple')
    plt.plot(plot_df['Date'], plot_df['lstm_pred'], label='LSTM Forecast', linestyle='-', color='orange')
    plt.title(f'DLT vs Prophet vs Silverkite vs Sarima vs LSTM - {label}')
    plt.xlabel('Date')
    plt.ylabel('Value')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


    # ===== Evaluation =====
    #merged_clean = merged.dropna()
    evaluate_performance(merged['y'], merged['dlt_pred'], f'DLT - {label}')
    evaluate_performance(merged['y'], merged['prophet_pred'], f'Prophet - {label}')
    evaluate_performance(merged['y'], merged['silverkite_pred'], f'Silverkite - {label}')
    evaluate_performance(merged['y'], merged['sarima_pred'], f'SARIMA - {label}')
    evaluate_performance(merged['y'], merged['lstm_pred'], f'LSTM - {label}')

    # ===== Iteration summary =====
    summary_df = pd.DataFrame(iteration_log)
    print("\nTrain/Test Ranges by Iteration:")
    print(summary_df.to_string(index=False))


We now apply the function above considering a three month test period and one day ahead prediction (forecast_horizon=1). Note that you will need to be patient with running this analysis since we are fitting many models (one for each period or day in this case).

In [ ]:
run_expanding_cv_forecast_mult(df, forecast_horizon=7, test_period_months=3, label="Weekly (3 months)")

For further illustration, we apply the function above considering a three month test period and one week ahead prediction (forecast_horizon=7).

In [ ]:
run_expanding_cv_forecast_mult(df, forecast_horizon=1, test_period_months=3, label="Daily (3 months)")

## Method 2: Random Forecast Periods

The disadvantage of using the CV approach in the first method is that it applies only to recent data, which might have similar characteristics in terms of variability, trend, seasonality and so forth. However, such characteristics will change over time and thus we might want to select a method that performs well under different charateristics of the time series. To do so, we can select the prediction periods randomly. We illustrate this idea below by predicting 12 weeks randomly selected. This is tobe compared with the CV approach where the forecast_horizon=7 and the test period was the last three months.

The function below implements this idea with the two important inputs num_weeks being the number of weeks to be predicted and the forecast_horizon being how the prediction to be performed for each week.

In [ ]:
import random

def run_random_weekly_forecast_mult_model(df, num_weeks=12, forecast_horizon=7, label="Random 12 Weeks"):
    df = df.sort_values('Date').reset_index(drop=True)
    df['Date'] = pd.to_datetime(df['Date'])
    max_date = df['Date'].max()
    cutoff_date = max_date - pd.DateOffset(years=3)

    eligible_indices = df[df['Date'] >= cutoff_date].index
    eligible_indices = [i for i in eligible_indices if i + forecast_horizon < len(df)]

    selected_indices = sorted(random.sample(eligible_indices, num_weeks))

    sarima_preds = []
    prophet_preds = []
    dlt_preds = []
    silverkite_preds = []
    logs = []

    for idx in selected_indices:
        train_df = df.iloc[:idx].copy()
        predict_window = df.iloc[idx : idx + forecast_horizon].copy()

        print(f"Predicting week: {predict_window['Date'].min().date()} ~ {predict_window['Date'].max().date()} (train up to {train_df['Date'].max().date()})")

        # --- Orbit DLT Model Implementation ---
        dlt = DLT(
            seasonality=None,
            response_col='y',
            date_col='Date',
            estimator='stan-map',
            seed=8888,
            global_trend_option='loglinear',
            n_bootstrap_draws=100,
            prediction_percentiles=[5, 95],
        )
        dlt.fit(train_df)
        dlt_forecast = dlt.predict(predict_window)
        dlt_preds.append(dlt_forecast[['Date', 'prediction']].rename(columns={'prediction': 'dlt_pred'}))

        # --- Prophet Model Implementation---
        changepoints = ['2020-03-12', '2020-12-16', '2021-05-19', '2021-11-10','2022-05-09']
        prophet_train = train_df.rename(columns={'Date': 'ds', 'y': 'y'})
        prophet_test = predict_window[['Date']].rename(columns={'Date': 'ds'})
        m = Prophet(
            yearly_seasonality=False,
            weekly_seasonality=False,
            changepoints=changepoints
        )
        m.fit(prophet_train)
        forecast = m.predict(prophet_test)
        prophet_forecast = forecast[['ds', 'yhat']].rename(columns={'ds': 'Date', 'yhat': 'prophet_pred'})
        prophet_preds.append(prophet_forecast)

        # --- SARIMA Model Implementation ---
        sarima_train = train_df.set_index("Date")["y"]
        try:
            sarima_model = SARIMAX(
                sarima_train,
                # order=(2, 1, 0),
                # seasonal_order=(1, 1, 1, 12),
                order=(1, 1, 0),
                seasonal_order=(1, 0, 1, 12),
                enforce_stationarity=False,
                enforce_invertibility=False,
                validate_specification=False
            ).fit(disp=False)

            sarima_forecast = sarima_model.forecast(steps=forecast_horizon)
            sarima_pred_df = predict_window[['Date']].copy()
            sarima_pred_df['sarima_pred'] = sarima_forecast.values
            sarima_preds.append(sarima_pred_df)
        except Exception as e:
            print(f"Failed to fit SARIMA for week starting {predict_window['Date'].min()}: {e}")


        # --- Silverkite Model Implementation ---
        df_combined_cp = train_df.rename(columns={"Date": "ts"})
        silverkite_config = ForecastConfig(
            model_template="SILVERKITE",
            forecast_horizon=forecast_horizon,
            coverage=0.95,
            metadata_param=MetadataParam(time_col="ts", value_col="y", freq="D"),
            model_components_param=ModelComponentsParam(
                growth={"growth_term": "quadratic"},
                changepoints={"changepoints_dict": {
                    "method": "auto",
                    "regularization_strength": 0.5,
                    "no_changepoint_proportion_from_end": 0.0,
                    "potential_changepoint_n": 20,
                    "resample_freq": "7D"
                },
                "seasonality_changepoints_dict": dict(
                    potential_changepoint_distance="30D",
                    seasonality_components_df=pd.DataFrame({
                        "name": ["tow", "conti_year"],
                        "period": [7.0, 1.0],
                        "order": [4, 6],
                        "seas_names": ["weekly", "yearly"]})
                )},
                seasonality={
                    "yearly_seasonality": True,
                    "quarterly_seasonality": False,
                    "monthly_seasonality": False,
                    "weekly_seasonality": False,
                    "daily_seasonality": False,
                },
                autoregression={"autoreg_dict": {"lag_dict": {"orders": [1, 2, 3, 7]}}},
                uncertainty={"uncertainty_dict": {"uncertainty_method": "simple_conditional_residuals"}},
                custom={
                    "feature_sets_enabled": True,
                    "extra_pred_cols": [],
                    "fit_algorithm_dict": {"fit_algorithm": "ridge"}
                }
            )
        )
        forecaster = Forecaster()
        silverkite_result = forecaster.run_forecast_config(df=df_combined_cp, config=silverkite_config)
        sk_df = silverkite_result.forecast.df.rename(columns={"ts": "Date", "forecast": "silverkite_pred"})
        silverkite_preds.append(sk_df[['Date', 'silverkite_pred']])

        # log
        logs.append({
            "Train End": train_df['Date'].max().date(),
            "Predict Start": predict_window['Date'].min().date(),
            "Predict End": predict_window['Date'].max().date()
        })

    # --- Merge predictions from all models ---
    dlt_result = pd.concat(dlt_preds, ignore_index=True)
    prophet_result = pd.concat(prophet_preds, ignore_index=True)
    silverkite_result = pd.concat(silverkite_preds, ignore_index=True)
    sarima_result = pd.concat(sarima_preds, ignore_index=True)

    merged = df[['Date', 'y']]
    merged = pd.merge(merged, dlt_result, on='Date', how='inner')
    merged = pd.merge(merged, prophet_result, on='Date', how='inner')
    merged = pd.merge(merged, silverkite_result, on='Date', how='inner')
    merged = pd.merge(merged, sarima_result, on='Date', how='inner')

    # --- Graph ---
    plt.figure(figsize=(14, 6))
    plt.plot(df['Date'], df['y'], label='Actual', color='black', linewidth=2)
    plt.plot(merged['Date'], merged['dlt_pred'], 'bo', label='DLT')
    plt.plot(merged['Date'], merged['prophet_pred'], 'go', label='Prophet')
    plt.plot(merged['Date'], merged['silverkite_pred'], 'ro', label='Silverkite')
    plt.plot(merged['Date'], merged['sarima_pred'], 'mo', label='SARIMA')

    plt.title(f'DLT vs Prophet vs Silverkite vs SARIMA - {label}')
    plt.xlabel('Date')
    plt.ylabel('Bitcoin Price (y)')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


    # --- Model evaluate ---
    evaluate_performance(merged['y'], merged['dlt_pred'], f"DLT - {label}")
    evaluate_performance(merged['y'], merged['prophet_pred'], f"Prophet - {label}")
    evaluate_performance(merged['y'], merged['silverkite_pred'], f"Silverkite - {label}")
    evaluate_performance(merged['y'], merged['sarima_pred'], f"SARIMA - {label}")


    # --- output ---
    summary_df = pd.DataFrame(logs)
    print("\nRandom Forecast Ranges:")
    print(summary_df.to_string(index=False))


# call function
# df = your bitcoin price dataframe with ['Date', 'y']
run_random_weekly_forecast_mult_model(df, num_weeks=12, forecast_horizon=7, label="12 Random Weeks")